# 🔬 Polarscope — X-ray your Polars DataFrames

**Polarscope** gives you deep, one-call inspection and cleaning for [Polars](https://pola.rs) DataFrames:

- **`ps.xray(df)`** — a full statistical X-ray of every column: distributions, missingness, outliers, data-quality flags, normality tests, inline histograms.
- **`ps.fix(df)`** — clean & optimize in one call: snake_case headers, trimmed strings, shrunk dtypes, and a report of what changed.
- **Plots** — correlation heatmaps, missing-value bars, distributions, categorical frequencies (plotly or altair).
- **Three bundled datasets** to play with: `titanic`, `diabetes`, and `cardio` (70,000 rows).

```
pip install polarscope            # core
pip install "polarscope[all]"     # + plotly, altair, scipy extras
```

In [1]:
# !pip install "polarscope[all]"   # uncomment on Kaggle/Colab

import polars as pl
import polarscope as ps

print("polars", pl.__version__, "| polarscope", ps.__version__)

polars 1.42.0 | polarscope 1.9.2


---
## 1 | Built-in datasets

Three datasets ship inside the package — no downloads, no paths:

| Loader | Rows × Cols | Contents |
|---|---|---|
| `ps.titanic()` | 156 × 12 | Mixed types, missing values — great for cleaning demos |
| `ps.diabetes()` | 768 × 9 | All numeric — great for statistics and correlations |
| `ps.cardio()` | 70,000 × 13 | Large all-numeric health records — great for performance |

In [2]:
df_titanic = ps.titanic()
df_diabetes = ps.diabetes()
df_cardio = ps.cardio()

for name, d in [("titanic", df_titanic), ("diabetes", df_diabetes), ("cardio", df_cardio)]:
    print(f"{name:9s} {d.shape}")

titanic   (156, 12)
diabetes  (768, 9)
cardio    (70000, 13)


---
## 2 | `ps.xray()` — the core inspection function

The default call analyzes **numeric columns** and returns a formatted
[Great Tables](https://posit-dev.github.io/great-tables/) view with essential statistics
and an inline distribution histogram per column.

In [3]:
ps.xray(df_diabetes)

GT(_tbl_data=shape: (9, 16)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬───────────┬────────────┐
│ column     ┆ dtype   ┆ count ┆ null_count ┆ … ┆ pct_missin ┆ n_outliers ┆ skew      ┆ distributi │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ g          ┆ ---        ┆ ---       ┆ on_plot    │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ ---        ┆ i64        ┆ f64       ┆ ---        │
│            ┆         ┆       ┆            ┆   ┆ f64        ┆            ┆           ┆ list[f64]  │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪═══════════╪════════════╡
│ Pregnancie ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 4          ┆ 0.899912  ┆ [246.0,    │
│ s          ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 103.0, …   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 1.0]       │
│ Glucose    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 5          ┆ 0.173414  ┆ [5.0, 0.0, │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ … 35.0]    │
│ BloodPress ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 45         ┆ -1.840005 ┆ [35.0,     │
│ ure        ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 0.0, …     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 2.0]       │
│ SkinThickn ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 1          ┆ 0.109159  ┆ [231.0,    │
│ ess        ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 55.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 1.0]       │
│ Insulin    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 35         ┆ 2.26781   ┆ [456.0,    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 148.0, …   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 1.0]       │
│ BMI        ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 19         ┆ -0.428143 ┆ [11.0,     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 0.0, …     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 1.0]       │
│ DiabetesPe ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 29         ┆ 1.916159  ┆ [266.0,    │
│ digreeFunc ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 205.0, …   │
│ tion       ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 3.0]       │
│ Age        ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 9          ┆ 1.127389  ┆ [300.0,    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 141.0, …   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 1.0]       │
│ Outcome    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0        ┆ 0          ┆ 0.633776  ┆ [500.0,    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 0.0, …     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆           ┆ 268.0]     │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴───────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x111a0eba0>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='center', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label=

### 2.1 Analyze every column with `include='all'`

String columns get their own statistics: top value and frequency, and the
min / median / avg / max value length. Boolean columns are analyzed as 0/1
(Mean = share of True) and temporal columns report their earliest/latest timestamps.

In [4]:
ps.xray(df_titanic, include='all')

GT(_tbl_data=shape: (12, 22)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ column     ┆ dtype   ┆ count ┆ null_count ┆ … ┆ median_len ┆ avg_length ┆ max_length ┆ distribut │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ gth        ┆ ---        ┆ ---        ┆ ion_plot  │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ ---        ┆ f64        ┆ i64        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆ f64        ┆            ┆            ┆ list[f64] │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ PassengerI ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ null       ┆ [13.0,    │
│ d          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 13.0, …   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 13.0]     │
│ Survived   ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ null       ┆ [102.0,   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 0.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 54.0]     │
│ Pclass     ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ null       ┆ [30.0,    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 0.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 96.0]     │
│ Name       ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 26.0       ┆ 27.217949  ┆ 57         ┆ [1.0,     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 1.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 1.0]      │
│ Sex        ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 4.0        ┆ 4.717949   ┆ 6          ┆ [100.0,   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 56.0]     │
│ …          ┆ …       ┆ …     ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …         │
│ Parch      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ null       ┆ [121.0,   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 0.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 2.0]      │
│ Ticket     ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 6.0        ┆ 6.961538   ┆ 18         ┆ [2.0,     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 2.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 1.0]      │
│ Fare       ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ null       ┆ [113.0,   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 18.0, …   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 3.0]      │
│ Cabin      ┆ String  ┆ 31    ┆ 125        ┆ … ┆ 3.0        ┆ 3.903226   ┆ 11         ┆ [2.0,     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 2.0, …    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 1.0]      │
│ Embarked   ┆ String  ┆ 155   ┆ 1          ┆ … ┆ 1.0        ┆ 1.0        ┆ 1          ┆ [110.0,   │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 32.0,     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ 13.0]     │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x10a41c050>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTy

### 2.2 Expanded mode — everything

`expanded=True` adds quantiles, kurtosis/MAD, uniqueness and duplicate ratios,
normality/uniformity tests (scipy), optimal-dtype suggestions, and a per-column
**quality flag** that combines missingness, constancy, skew, outliers, and normality
into a single `✓ OK` / `⚠ SHAKY` verdict.

In [5]:
ps.xray(df_titanic, include='all', expanded=True)

GT(_tbl_data=shape: (12, 41)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ column     ┆ dtype   ┆ count ┆ null_count ┆ … ┆ top_3      ┆ sample_val ┆ shakiness_ ┆ quality_f │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---        ┆ s          ┆ score      ┆ lag       │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ str        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆            ┆ str        ┆ i64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ PassengerI ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ 2          ┆ ⚠ SHAKY   │
│ d          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Survived   ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ 1          ┆ ✓ OK      │
│ Pclass     ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ 1          ┆ ✓ OK      │
│ Name       ┆ String  ┆ 156   ┆ 0          ┆ … ┆ Bonnell,   ┆ Bonnell,   ┆ 1          ┆ ✓ OK      │
│            ┆         ┆       ┆            ┆   ┆ Miss.      ┆ Miss.      ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ El... (1), ┆ El...,     ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ Cumi…      ┆ Andersso…  ┆            ┆           │
│ Sex        ┆ String  ┆ 156   ┆ 0          ┆ … ┆ male       ┆ male,      ┆ 0          ┆ ✓ OK      │
│            ┆         ┆       ┆            ┆   ┆ (100),     ┆ female     ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ female     ┆            ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ (56)       ┆            ┆            ┆           │
│ …          ┆ …       ┆ …     ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …         │
│ Parch      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ 4          ┆ ⚠ SHAKY   │
│ Ticket     ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 113803     ┆ 113803,    ┆ 0          ┆ ✓ OK      │
│            ┆         ┆       ┆            ┆   ┆ (2), 19950 ┆ 231919,    ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ (2),       ┆ 14311      ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ 349909 …   ┆            ┆            ┆           │
│ Fare       ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ null       ┆ null       ┆ 4          ┆ ⚠ SHAKY   │
│ Cabin      ┆ String  ┆ 31    ┆ 125        ┆ … ┆ C123 (2),  ┆ C123, F33, ┆ 1          ┆ ✓ OK      │
│            ┆         ┆       ┆            ┆   ┆ C23 C25    ┆ C52        ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ C27 (2),   ┆            ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ D26…       ┆            ┆            ┆           │
│ Embarked   ┆ String  ┆ 155   ┆ 1          ┆ … ┆ S (110), C ┆ S, C, Q    ┆ 0          ┆ ✓ OK      │
│            ┆         ┆       ┆            ┆   ┆ (32), Q    ┆            ┆            ┆           │
│            ┆         ┆       ┆            ┆   ┆ (13)       ┆            ┆            ┆           │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x1285e1480>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='center', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='center', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', colum

### 2.3 String-heavy data

For String / Categorical / Enum columns the expanded view adds the **mode share**
(how dominant the top value is), the **top-3 values**, and a **sample** across the
frequency range — so mostly-text frames are first-class citizens too.

In [6]:
df_strings = pl.DataFrame({
    "city": ["Oslo", "Bergen", "Oslo", "Oslo", "Trondheim", "Bergen", "Oslo", "Stavanger"] * 25,
    "segment": pl.Series(["Private", "Business", "Private", "Public"] * 50, dtype=pl.Categorical),
    "revenue": [1200.5, 880.0, 1500.25, 990.0] * 50,
})
ps.xray(df_strings, include='all', expanded=True)

GT(_tbl_data=shape: (3, 41)
┌─────────┬────────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ column  ┆ dtype      ┆ count ┆ null_count ┆ … ┆ normality_ ┆ uniformity ┆ shakiness_ ┆ quality_f │
│ ---     ┆ ---        ┆ ---   ┆ ---        ┆   ┆ test       ┆ _test      ┆ score      ┆ lag       │
│ str     ┆ str        ┆ i64   ┆ i64        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│         ┆            ┆       ┆            ┆   ┆ str        ┆ str        ┆ i64        ┆ str       │
╞═════════╪════════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ city    ┆ String     ┆ 200   ┆ 0          ┆ … ┆ N/A (non-n ┆ N/A (non-n ┆ 0          ┆ ✓ OK      │
│         ┆            ┆       ┆            ┆   ┆ umeric)    ┆ umeric)    ┆            ┆           │
│ segment ┆ Categorica ┆ 200   ┆ 0          ┆ … ┆ N/A (non-n ┆ N/A (non-n ┆ 0          ┆ ✓ OK      │
│         ┆ l          ┆       ┆            ┆   ┆ umeric)    ┆ umeric)    ┆            ┆           │
│ revenue ┆ Float64    ┆ 200   ┆ 0          ┆ … ┆ NON-NORMAL ┆ NON-UNIFOR ┆ 1          ┆ ✓ OK      │
│         ┆            ┆       ┆            ┆   ┆ (Shapiro-W ┆ M (KS,     ┆            ┆           │
│         ┆            ┆       ┆            ┆   ┆ ilk, p=0.… ┆ p=0.000)   ┆            ┆           │
└─────────┴────────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x12bcfbe50>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='center', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='center', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='min', type=<ColInfoTypeEnum.default: 1>, column_label='min', column_align='center', column_width=None), ColInfo(var='max', type=<ColInfoTypeEnum.default: 1>, column_label='max', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='pct_missing', type=<ColInfoTypeEnum.default: 1>, column_label='pct_missing', column_align='center', column_width=None), ColInfo(var='n_unique', type=<ColInfoTypeEnum.default: 1>, column_label='n_unique', column_align='center', column_width=None), ColInfo(var='uniqueness_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='uniqueness_ratio', column_align='center', column_width=None), ColInfo(var='n_duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='n_duplicates', column_align='center', column_width=None), ColInfo(var='pct_duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='pct_duplicates', column_align='center', column_width=None), ColInfo(var='n_zero', type=<ColInfoTypeEnum.default: 1>, column_label='n_zero', column_align='center', column_width=None), ColInfo(var='pct_zero', type=<ColInfoTypeEnum.default: 1>, column_label='pct_zero', column_align='center', column_width=None), ColInfo(var='pct_pos', type=<ColInfoTypeEnum.default: 1>, column_label='pct_pos', column_align='center', column_width=None), ColInfo(var='pct_neg', type=<ColInfoTypeEnum.default: 1>, column_label='pct_neg', column_align='center', column_width=None), ColInfo(var='top', type=<ColInfoTypeEnum.default: 1>, column_label='top', column_align='left', column_width=None), ColInfo(var='top_freq', type=<ColInfoTypeEnum.default: 1>, column_label='top_freq', column_align='center', column_width=None), ColInfo(var='min_length', type=<ColInfoTypeEnum.default: 1>, column_label='min_length', column_ali

### 2.4 Correlation with a target + model usability

- `corr_target=` adds a correlation column (with a ±1-scaled bar) against any numeric target.
- `model_usability=True` scores each column 0–100 for ML readiness with flags
  (ID-like, constant, extreme outliers/skew, high missingness, …) and a plain-text recommendation.

In [7]:
ps.xray(df_diabetes, corr_target='Outcome', model_usability=True)

GT(_tbl_data=shape: (9, 21)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ column     ┆ dtype   ┆ count ┆ null_count ┆ … ┆ correlatio ┆ usability_ ┆ usability_ ┆ recommend │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ n_plot     ┆ flags      ┆ score      ┆ ation     │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆ f64        ┆ str        ┆ f64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ Pregnancie ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.221898   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│ s          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ Glucose    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.466581   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ BloodPress ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.065068   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│ ure        ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ SkinThickn ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.074752   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│ ess        ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ Insulin    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.130548   ┆ EK,NN,UC   ┆ 84.745763  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ BMI        ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.292695   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ DiabetesPe ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.173844   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│ digreeFunc ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ tion       ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Age        ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.238356   ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ Outcome    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ null       ┆ BN,NN,UC   ┆ 88.135593  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x12bdcce50>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='center', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='min', type=<ColInfoTypeEnum.default: 1>, column_label='min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column

### 2.5 Big data? No problem

The whole X-ray of the 70,000-row cardio dataset takes well under a second —
the header of every table shows the exact timing.

In [8]:
ps.xray(df_cardio, include='all', expanded=True)

GT(_tbl_data=shape: (13, 32)
┌────────┬─────────┬───────┬────────────┬───┬────────────┬─────────────┬─────────────┬─────────────┐
│ column ┆ dtype   ┆ count ┆ null_count ┆ … ┆ n_outliers ┆ pct_outlier ┆ shakiness_s ┆ quality_fla │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---        ┆ s           ┆ core        ┆ g           │
│ str    ┆ str     ┆ i64   ┆ i64        ┆   ┆ i64        ┆ ---         ┆ ---         ┆ ---         │
│        ┆         ┆       ┆            ┆   ┆            ┆ f64         ┆ i64         ┆ str         │
╞════════╪═════════╪═══════╪════════════╪═══╪════════════╪═════════════╪═════════════╪═════════════╡
│ id     ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 0          ┆ 0.0         ┆ 2           ┆ ⚠ SHAKY     │
│ age    ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 4          ┆ 0.005714    ┆ 1           ┆ ✓ OK        │
│ gender ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 0          ┆ 0.0         ┆ 1           ┆ ✓ OK        │
│ height ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 519        ┆ 0.741429    ┆ 2           ┆ ⚠ SHAKY     │
│ weight ┆ Float64 ┆ 70000 ┆ 0          ┆ … ┆ 1819       ┆ 2.598571    ┆ 1           ┆ ✓ OK        │
│ …      ┆ …       ┆ …     ┆ …          ┆ … ┆ …          ┆ …           ┆ …           ┆ …           │
│ gluc   ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 10521      ┆ 15.03       ┆ 3           ┆ ⚠ SHAKY     │
│ smoke  ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 6169       ┆ 8.812857    ┆ 3           ┆ ⚠ SHAKY     │
│ alco   ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 3764       ┆ 5.377143    ┆ 4           ┆ ⚠ SHAKY     │
│ active ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 13739      ┆ 19.627143   ┆ 2           ┆ ⚠ SHAKY     │
│ cardio ┆ Int64   ┆ 70000 ┆ 0          ┆ … ┆ 0          ┆ 0.0         ┆ 1           ┆ ✓ OK        │
└────────┴─────────┴───────┴────────────┴───┴────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12be18590>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='center', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='center', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='min', type=<ColInfoTypeEnum.default: 1>, column_label='min', column_align='center', column_width=None), ColInfo(var='max', type=<ColInfoTypeEnum.default: 1>, column_label='max', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='pct_missing', type=<ColInfoTypeEnum.default: 1>, column_label='pct_missing', column_align='center', column_width=None), ColInfo(var='n_unique', type=<ColInfoTypeEnum.default: 1>, column_label='n_unique', column_align='center', column_width=None), ColInfo(var='uniqueness_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='uniqueness_ratio', column_align='center', column_width=None), ColInfo(var='n_duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='n_duplicates', column_align='center', column_width=None), ColInfo(var='pct_duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='pct_duplicates', column_align='center', column_width=None), ColInfo(var='n_zero', type=<ColInfoTypeEnum.default: 1>, column_label='n_zero', column_align='center', column_width=None), ColInfo(var='pct_zero', type=<ColInfoTypeEnum.default: 1>, column_label='pct_zero', column_align='center', column_width=None), ColInfo(var='pct_pos', type=<ColInfoTypeEnum.default: 1>, column_label='pct_pos', column_align='center', column_width=None), ColInfo(var='pct_neg', type=<ColInfoTypeEnum.default: 1>, co

### 2.6 Plain DataFrame output

Set `great_tables=False` to get the summary as a regular Polars DataFrame —
useful for programmatic checks or piping into your own reports.

In [9]:
summary = ps.xray(df_diabetes, great_tables=False)
summary.select(["column", "mean", "std", "pct_missing", "n_outliers", "skew"])

column,mean,std,pct_missing,n_outliers,skew
str,f64,f64,f64,i64,f64
"""Pregnancies""",3.845052,3.369578,0.0,4,0.899912
"""Glucose""",120.894531,31.972618,0.0,5,0.173414
"""BloodPressure""",69.105469,19.355807,0.0,45,-1.840005
"""SkinThickness""",20.536458,15.952218,0.0,1,0.109159
"""Insulin""",79.799479,115.244002,0.0,35,2.26781
"""BMI""",31.992578,7.88416,0.0,19,-0.428143
"""DiabetesPedigreeFunction""",0.471876,0.331329,0.0,29,1.916159
"""Age""",33.240885,11.760232,0.0,9,1.127389
"""Outcome""",0.348958,0.476951,0.0,0,0.633776


---
## 3 | `ps.fix()` — clean & optimize in one call

`fix()` runs the safe, **lossless** cleanups by default and prints a report:

| Step | Default | What it does |
|---|---|---|
| `case="snake"` | on | Clean column names (`"Kunde Navn"` → `kunde_navn`); also `camel`, `pascal`, `kebab`, `upper`, `lower`, or `None` |
| `strip_strings` | on | Trim whitespace; empty/whitespace-only strings → null |
| `shrink_dtypes` | on | Downcast ints/floats, low-cardinality strings → Categorical |
| `drop_empty_columns` | on | Remove 100%-null columns |
| `drop_duplicate_rows` | **off** | Remove exact duplicate rows |
| `missing_threshold` | **off** | Drop columns above a missing-share threshold |
| `drop_constant_columns` | **off** | Drop single-value columns |
| `outliers` | **off** | Null out extreme values (`"iqr"` / `"zscore"`) |

Anything that removes or alters data is opt-in. The input frame is never mutated.

In [10]:
df_messy = pl.DataFrame({
    "Kunde Navn": ["  Alice ", "Bob", "", "Bob", None],
    "Årlig Inntekt (kr)": [50_000, 60_000, 55_000, 60_000, None],
    "myColumnName": [1.0, 2.0, 3.0, 2.0, 1.0],
    "empty_col": pl.Series([None] * 5, dtype=pl.String),
    "const": ["x", "x", "x", "x", "x"],
})
df_messy

Kunde Navn,Årlig Inntekt (kr),myColumnName,empty_col,const
str,i64,f64,str,str
""" Alice """,50000,1.0,null,"""x"""
"""Bob""",60000,2.0,null,"""x"""
"""""",55000,3.0,null,"""x"""
"""Bob""",60000,2.0,null,"""x"""
null,null,1.0,null,"""x"""


In [11]:
df_fixed = ps.fix(df_messy)
df_fixed

ps.fix report
─────────────
Renamed 3 column(s) to snake case
Stripped whitespace in 3 string column(s); 1 value(s) trimmed, 1 empty string(s) -> null
Dropped 1 empty column(s): empty_col
Shrunk dtypes in 4 column(s): kunde_navn: String -> Categorical; arlig_inntekt_kr: Int64 -> Int32; my_column_name: Float64 -> Float32; const: String -> Categorical
Memory: 0.1 KB -> 0.1 KB (18% saved)
Rows: 5 (unchanged) | Columns: 5 -> 4


kunde_navn,arlig_inntekt_kr,my_column_name,const
cat,i32,f32,cat
"""Alice""",50000,1.0,"""x"""
"""Bob""",60000,2.0,"""x"""
null,55000,3.0,"""x"""
"""Bob""",60000,2.0,"""x"""
null,null,1.0,"""x"""


### 3.1 Pick your naming style with `case=`

In [12]:
for style in ["snake", "camel", "pascal", "kebab", "upper", "lower"]:
    cols = ps.fix(df_messy, case=style, verbose=False).columns
    print(f"{style:7s} -> {cols}")

snake   -> ['kunde_navn', 'arlig_inntekt_kr', 'my_column_name', 'const']
camel   -> ['kundeNavn', 'arligInntektKr', 'myColumnName', 'const']
pascal  -> ['KundeNavn', 'ArligInntektKr', 'MyColumnName', 'Const']
kebab   -> ['kunde-navn', 'arlig-inntekt-kr', 'my-column-name', 'const']
upper   -> ['KUNDE_NAVN', 'ARLIG_INNTEKT_KR', 'MYCOLUMNNAME', 'CONST']
lower   -> ['kunde_navn', 'arlig_inntekt_kr', 'mycolumnname', 'const']


### 3.2 Opt in to deeper cleaning

In [13]:
ps.fix(
    df_messy,
    drop_duplicate_rows=True,     # exact duplicates out
    drop_constant_columns=True,   # single-value columns out
    missing_threshold=0.3,        # drop columns >30% missing
)

ps.fix report
─────────────
Renamed 3 column(s) to snake case
Stripped whitespace in 3 string column(s); 1 value(s) trimmed, 1 empty string(s) -> null
Dropped 1 empty column(s): empty_col
Dropped 1 column(s) >30% missing: kunde_navn
Dropped 1 constant column(s): const
Removed 1 duplicate row(s)
Shrunk dtypes in 2 column(s): arlig_inntekt_kr: Int64 -> Int32; my_column_name: Float64 -> Float32
Memory: 0.1 KB -> 0.0 KB (67% saved)
Rows: 5 -> 4 | Columns: 5 -> 2


arlig_inntekt_kr,my_column_name
i32,f32
50000,1.0
60000,2.0
55000,3.0
null,1.0


### 3.3 On real data: titanic shrinks ~40% with pure defaults

In [14]:
df_clean = ps.fix(df_titanic)
df_clean.head(5)

ps.fix report
─────────────
Renamed 12 column(s) to snake case
Stripped whitespace in 5 string column(s); 1 value(s) trimmed, 0 empty string(s) -> null
Shrunk dtypes in 8 column(s): passenger_id: Int64 -> Int16; survived: Int64 -> Int8; pclass: Int64 -> Int8; sex: String -> Categorical; age: Float64 -> Float32; sib_sp: Int64 -> Int8; parch: Int64 -> Int8; embarked: String -> Categorical
Memory: 14.7 KB -> 9.3 KB (37% saved)
Rows: 156 (unchanged) | Columns: 12 (unchanged)


passenger_id,survived,pclass,name,sex,age,sib_sp,parch,ticket,fare,cabin,embarked
i16,i8,i8,str,cat,f32,i8,i8,str,f64,str,cat
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


---
## 4 | Plots

All plotting functions accept `backend="plotly"` (interactive, default) or `backend="altair"`.

### 4.1 Correlation heatmap — full matrix or against a target

In [15]:
ps.corr_heatmap(df_diabetes)

In [16]:
ps.corr_heatmap(df_diabetes, target='Outcome')

### 4.2 Missing values per column

In [17]:
ps.missingval_plot(df_titanic, normalize=True)

### 4.3 Distribution of a single column

In [18]:
ps.dist_plot(df_diabetes, column='Glucose', bins=40)

### 4.4 Categorical frequencies — top and bottom categories

In [19]:
ps.cat_plot(df_titanic, top=5, bottom=3)

### 4.5 Clustered correlation plot (requires scipy)

In [20]:
ps.corr_plot(df_diabetes, clustered=True, method='spearman')

---
## 5 | The 3-line EDA workflow

Load → fix → xray. That's the whole pipeline for a first look at any dataset.

In [21]:
df = ps.cardio()                                   # 1. load (or pl.read_csv/parquet)
df = ps.fix(df)                                    # 2. clean + optimize
ps.xray(df, include='all', corr_target='cardio')   # 3. inspect

ps.fix report
─────────────
Shrunk dtypes in 12 column(s): id: Int64 -> Int32; age: Int64 -> Int16; gender: Int64 -> Int8; height: Int64 -> Int16; ap_hi: Int64 -> Int16; ap_lo: Int64 -> Int16; cholesterol: Int64 -> Int8; gluc: Int64 -> Int8; smoke: Int64 -> Int8; alco: Int64 -> Int8; active: Int64 -> Int8; cardio: Int64 -> Int8
Memory: 6.9 MB -> 1.8 MB (74% saved)
Rows: 70,000 (unchanged) | Columns: 13 (unchanged)


GT(_tbl_data=shape: (13, 18)
┌────────┬─────────┬───────┬────────────┬───┬───────────┬──────────────┬─────────────┬─────────────┐
│ column ┆ dtype   ┆ count ┆ null_count ┆ … ┆ skew      ┆ distribution ┆ correlation ┆ correlation │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---       ┆ _plot        ┆ ---         ┆ _plot       │
│ str    ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64       ┆ ---          ┆ f64         ┆ ---         │
│        ┆         ┆       ┆            ┆   ┆           ┆ list[f64]    ┆             ┆ f64         │
╞════════╪═════════╪═══════╪════════════╪═══╪═══════════╪══════════════╪═════════════╪═════════════╡
│ id     ┆ Int32   ┆ 70000 ┆ 0          ┆ … ┆ -0.001278 ┆ [5874.0,     ┆ 0.003799    ┆ 0.003799    │
│        ┆         ┆       ┆            ┆   ┆           ┆ 5791.0, …    ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 5774.0]      ┆             ┆             │
│ age    ┆ Int16   ┆ 70000 ┆ 0          ┆ … ┆ -0.307049 ┆ [4.0, 0.0, … ┆ 0.238159    ┆ 0.238159    │
│        ┆         ┆       ┆            ┆   ┆           ┆ 7042.0]      ┆             ┆             │
│ gender ┆ Int8    ┆ 70000 ┆ 0          ┆ … ┆ 0.630947  ┆ [45530.0,    ┆ 0.008109    ┆ 0.008109    │
│        ┆         ┆       ┆            ┆   ┆           ┆ 0.0, …       ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 24470.0]     ┆             ┆             │
│ height ┆ Int16   ┆ 70000 ┆ 0          ┆ … ┆ -0.642174 ┆ [17.0, 7.0,  ┆ -0.010821   ┆ -0.010821   │
│        ┆         ┆       ┆            ┆   ┆           ┆ … 1.0]       ┆             ┆             │
│ weight ┆ Float64 ┆ 70000 ┆ 0          ┆ … ┆ 1.012048  ┆ [5.0, 126.0, ┆ 0.18166     ┆ 0.18166     │
│        ┆         ┆       ┆            ┆   ┆           ┆ … 2.0]       ┆             ┆             │
│ …      ┆ …       ┆ …     ┆ …          ┆ … ┆ …         ┆ …            ┆ …           ┆ …           │
│ gluc   ┆ Int8    ┆ 70000 ┆ 0          ┆ … ┆ 2.39741   ┆ [59479.0,    ┆ 0.089307    ┆ 0.089307    │
│        ┆         ┆       ┆            ┆   ┆           ┆ 0.0, …       ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 5331.0]      ┆             ┆             │
│ smoke  ┆ Int8    ┆ 70000 ┆ 0          ┆ … ┆ 2.905805  ┆ [63831.0,    ┆ -0.015486   ┆ -0.015486   │
│        ┆         ┆       ┆            ┆   ┆           ┆ 0.0, …       ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 6169.0]      ┆             ┆             │
│ alco   ┆ Int8    ┆ 70000 ┆ 0          ┆ … ┆ 3.956522  ┆ [66236.0,    ┆ -0.00733    ┆ -0.00733    │
│        ┆         ┆       ┆            ┆   ┆           ┆ 0.0, …       ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 3764.0]      ┆             ┆             │
│ active ┆ Int8    ┆ 70000 ┆ 0          ┆ … ┆ -1.52944  ┆ [13739.0,    ┆ -0.035653   ┆ -0.035653   │
│        ┆         ┆       ┆            ┆   ┆           ┆ 0.0, …       ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 56261.0]     ┆             ┆             │
│ cardio ┆ Int8    ┆ 70000 ┆ 0          ┆ … ┆ 0.0012    ┆ [35021.0,    ┆ null        ┆ null        │
│        ┆         ┆       ┆            ┆   ┆           ┆ 0.0, …       ┆             ┆             │
│        ┆         ┆       ┆            ┆   ┆           ┆ 34979.0]     ┆             ┆             │
└────────┴─────────┴───────┴────────────┴───┴───────────┴──────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f14c5d0>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='dtype', type=<ColInfoTypeEnum.default: 1>, column_label='dtype', column_align='center', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='center', column_width=Non

---
## Learn more

- **Install**: `pip install polarscope` — [PyPI](https://pypi.org/project/polarscope/)
- **Source & issues**: [github.com/pytoned/polarscope](https://github.com/pytoned/polarscope)
- **Docs in your editor**: every function has a full docstring — try `help(ps.xray)`

*Built on [Polars](https://pola.rs) and [Great Tables](https://posit-dev.github.io/great-tables/). Inspired by [klib](https://github.com/akanz1/klib).*